# B07 — Unified `/predict` Eval (Type 1 + Type 2)

Converts dataset samples into the unified request schema and sends them to `/predict`.

**Unified schema:**
```json
{
  "query_id": "T1_0001",
  "type": "type1",
  "query": "Is Student A eligible for graduation?",
  "premises": ["...", "..."],
  "options": ["Yes", "No", "Uncertain"]
}
```

- **Type 1** — `Logic_Based_Educational_Queries.json`; MCQ options embedded or as list; polar questions use `["Yes", "No", "Uncertain"]`.
- **Type 2** — `type2_physics_questions_NL_sample100.csv`; no premises/options; returns mock until type 2 is merged.
- Compares predicted answer against gold label and reports accuracy + latency.

In [46]:
import json, re, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx
import pandas as pd

API_BASE    = "https://api.iamphuckhang.dev"
PREDICT_URL = f"{API_BASE}/predict"

# --- sample sizes (set to None for full dataset) ---
N_TYPE1     = 20
N_TYPE2     = 10
CONCURRENCY = 8
TIMEOUT     = 180.0

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", PREDICT_URL)

dataset : /home/phuckhang/MyWorkspace/Exact2026/src/exact/datasets/exact
endpoint: https://api.iamphuckhang.dev/predict


## Build unified sample list

In [47]:
_OPTION_LINE = re.compile(r"^\s*([A-E])[.)]\s+(.+)$", re.MULTILINE)

def build_type1_samples(path: Path, n: int | None) -> list[dict]:
    raw = json.load(open(path))
    samples = []
    for g_idx, group in enumerate(raw):
        for q_idx, (question, gold) in enumerate(zip(group["questions"], group["answers"])):
            is_mcq = bool(_OPTION_LINE.search(question))
            # Send no options. MCQ options are embedded in the question body
            # (A./A) lines) and extracted server-side; polar questions use the
            # native YNU path (check_ynu) returning Yes/No/Uncertain directly.
            # Supplying ["Yes","No","Uncertain"] would force is_mcq=True server-side.
            samples.append({
                "query_id": f"T1_{g_idx:04d}_{q_idx:02d}",
                "type": "type1",
                "query": question,
                "premises": group["premises-NL"],
                "options": None,
                "_gold": gold,
                "_is_mcq": is_mcq,
            })
    return samples[:n] if n else samples

def build_type2_samples(path: Path, n: int | None) -> list[dict]:
    df = pd.read_csv(path)
    samples = []
    for _, row in df.iterrows():
        samples.append({
            "query_id": str(row["id"]),
            "type": "type2",
            "query": str(row["question"]),
            "premises": None,
            "options": None,
            "_gold": str(row["answer"]),
            "_unit": str(row.get("unit", "")),
        })
    return samples[:n] if n else samples

type1_samples = build_type1_samples(
    DATA / "Logic_Based_Educational_Queries.json", N_TYPE1
)
type2_samples = build_type2_samples(
    DATA / "type2_physics_questions_NL_sample100.csv", N_TYPE2
)
all_samples = type1_samples

t1_mcq  = sum(1 for s in type1_samples if s["_is_mcq"])
t1_ynu  = len(type1_samples) - t1_mcq
print(f"Type 1 : {len(type1_samples)} samples  (MCQ: {t1_mcq}, polar/YNU: {t1_ynu})")
print(f"Type 2 : {len(type2_samples)} samples")
print(f"Total  : {len(all_samples)} samples")
print()
print("Example type1 payload:")
ex = {k: v for k, v in type1_samples[0].items() if not k.startswith("_")}
print(json.dumps(ex, indent=2)[:600])

Type 1 : 20 samples  (MCQ: 10, polar/YNU: 10)
Type 2 : 10 samples
Total  : 20 samples

Example type1 payload:
{
  "query_id": "T1_0000_00",
  "type": "type1",
  "query": "Which conclusion follows with the fewest premises?\nA. If a Python project is not optimized, then it is not well-tested\nB. If all Python projects are optimized, then all Python projects are well-structured\nC. If a Python project is well-tested, then it must be clean and readable\nD. If a Python project is not optimized, then it does not follow PEP 8 standards",
  "premises": [
    "If a Python code is well-tested, then the project is optimized.",
    "If a Python code does not follow PEP 8 standards, then it is not well-tested.",
 


## Send to `/predict`

In [ ]:
def to_payload(sample: dict) -> dict:
    """Convert a sample to the unified /predict request body."""
    body: dict = {
        "query_id": sample["query_id"],
        "type": sample["type"],
        "query": sample["query"],
    }
    if sample["premises"]:
        body["premises"] = sample["premises"]
    if sample["options"]:
        body["options"] = sample["options"]
    return body


async def call(client, sem, sample):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(PREDICT_URL, json=to_payload(sample), timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
    return {
        **sample,
        "_response": body,
        "_latency": time.perf_counter() - t0,
        "_error": err,
    }


async def run_eval(samples):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(s):
            nonlocal done
            res = await call(client, sem, s)
            done += 1
            if done % 5 == 0 or done == len(samples):
                print(f"  {done}/{len(samples)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(s) for s in samples))


print(f"Sending {len(all_samples)} requests (concurrency={CONCURRENCY})...")
t0 = time.perf_counter()
results = await run_eval(all_samples)
wall = time.perf_counter() - t0

errors  = [r for r in results if r["_error"]]
success = [r for r in results if not r["_error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")

Sending 20 requests (concurrency=8)...


In [49]:
if errors:
    print("=== Errors ===")
    for r in errors[:5]:
        print(f"  [{r['query_id']}] {r['_error']}")

## Results per sample

In [52]:
for r in success:
    resp = r["_response"]
    pred = resp.get("answer", "?")
    gold = r["_gold"]
    ok   = "✓" if str(pred).strip().upper() == str(gold).strip().upper() else "✗"
    print(f"{ok} [{r['query_id']}] pred={pred!r:12s} gold={gold!r:10s}  "
          f"{resp.get('question_type','?'):6s}  {r['_latency']:.1f}s")
    if resp.get("error"):
        print(f"    !! {resp['error']}")

✗ [T1_0000_00] pred='Uncertain'  gold='A'         mcq     68.9s
✓ [T1_0000_01] pred='Yes'        gold='Yes'       ynu     64.1s
✗ [T1_0001_00] pred='Uncertain'  gold='C'         mcq     41.7s
✗ [T1_0001_01] pred='Uncertain'  gold='Yes'       ynu     42.1s
✗ [T1_0002_00] pred='Uncertain'  gold='C'         mcq     40.7s
✗ [T1_0002_01] pred='Uncertain'  gold='Yes'       ynu     41.6s
✗ [T1_0003_00] pred='Uncertain'  gold='A'         mcq     42.0s
✗ [T1_0003_01] pred='Uncertain'  gold='Yes'       ynu     40.8s
✗ [T1_0004_00] pred='Uncertain'  gold='C'         mcq     33.0s
✗ [T1_0004_01] pred='Uncertain'  gold='Yes'       ynu     33.8s
✗ [T1_0005_00] pred='Uncertain'  gold='B'         mcq     33.0s
✗ [T1_0005_01] pred='Uncertain'  gold='Yes'       ynu     33.2s
✗ [T1_0006_00] pred='Uncertain'  gold='C'         mcq     31.0s
✗ [T1_0006_01] pred='Uncertain'  gold='No'        ynu     32.6s
✗ [T1_0007_00] pred='Uncertain'  gold='B'         mcq     25.0s
✗ [T1_0007_01] pred='Uncertain'  gold='N

## Accuracy + metrics

In [53]:
t1_res = [r for r in success if r["type"] == "type1"]
t2_res = [r for r in success if r["type"] == "type2"]

def accuracy(rows):
    if not rows:
        return 0.0, 0, 0
    correct = sum(
        1 for r in rows
        if str(r["_response"].get("answer","")).strip().upper()
           == str(r["_gold"]).strip().upper()
    )
    return correct / len(rows), correct, len(rows)

# --- Type 1 ---
acc1, c1, n1 = accuracy(t1_res)
t1_mcq_res = [r for r in t1_res if r["_is_mcq"]]
t1_ynu_res = [r for r in t1_res if not r["_is_mcq"]]
acc1_mcq, c1_mcq, n1_mcq = accuracy(t1_mcq_res)
acc1_ynu, c1_ynu, n1_ynu = accuracy(t1_ynu_res)

print("=== Type 1 Accuracy ===")
print(f"  Overall : {acc1:.1%}  ({c1}/{n1})")
print(f"  MCQ     : {acc1_mcq:.1%}  ({c1_mcq}/{n1_mcq})")
print(f"  Polar   : {acc1_ynu:.1%}  ({c1_ynu}/{n1_ynu})")

# --- Type 1 answer distribution ---
ans_dist = Counter(r["_response"].get("answer") for r in t1_res)
print("\n=== Type 1 answer distribution ===")
for k, v in ans_dist.most_common():
    print(f"  {str(k):12s}: {v}")

# --- Type 1 solver diagnostics ---
solver_used = sum(
    1 for r in t1_res
    if r["_response"].get("routing_diagnostics", {}).get("solver_used")
)
print(f"\n=== Type 1 solver ===\n  solver_used: {solver_used}/{n1}")

# --- Type 2 (mock) ---
print(f"\n=== Type 2 (mock) ===")
print(f"  Responses: {len(t2_res)}  (all return mock 'Unknown' until pipeline is merged)")

# --- Latency ---
lat = [r["_latency"] for r in success]
print(f"\n=== Latency (all) ===")
print(f"  mean={statistics.mean(lat):.2f}s  p50={statistics.median(lat):.2f}s  max={max(lat):.2f}s")

=== Type 1 Accuracy ===
  Overall : 5.0%  (1/20)
  MCQ     : 0.0%  (0/10)
  Polar   : 10.0%  (1/10)

=== Type 1 answer distribution ===
  Uncertain   : 19
  Yes         : 1

=== Type 1 solver ===
  solver_used: 17/20

=== Type 2 (mock) ===
  Responses: 0  (all return mock 'Unknown' until pipeline is merged)

=== Latency (all) ===
  mean=34.90s  p50=33.11s  max=68.90s


In [54]:
# --- Diagnostic table: where does each type1 sample stop? -------------------
# Separates real logical uncertainty (Z3_TRUE_UNCERTAIN) from parser/verifier/
# mode gaps so we know what to fix next.
hdr = f"{'sample':14s} {'used':5s} {'verif':5s} {'supp':5s} {'mode':16s} {'opt_fol':7s} {'cause'}"
print(hdr)
print("-" * len(hdr))
from collections import Counter as _C
cause_counts = _C()
for r in [x for x in success if x["type"] == "type1"]:
    rd = r["_response"].get("routing_diagnostics") or {}
    qs = rd.get("query_spec") or {}
    n_opt_fol = sum(1 for c in qs.get("option_claims", []) if c.get("fol"))
    cause = rd.get("uncertainty_cause")
    cause_counts[cause] += 1
    print(f"{r['query_id']:14s} "
          f"{str(rd.get('solver_used')):5s} "
          f"{str(rd.get('premise_bundle_verified')):5s} "
          f"{str(qs.get('supported')):5s} "
          f"{str(qs.get('solver_mode')):16s} "
          f"{n_opt_fol:<7d} "
          f"{cause if cause is not None else '— (solved)'}")

print("\n=== uncertainty_cause distribution ===")
for c, n in cause_counts.most_common():
    print(f"  {str(c) if c is not None else '— (solved)':40s}: {n}")

# Premise warnings actually seen (non-blocking; solver still ran)
warn = _C()
for r in [x for x in success if x["type"] == "type1"]:
    for w in (r["_response"].get("routing_diagnostics") or {}).get("premise_warnings", []):
        warn[w.split(":")[0]] += 1
if warn:
    print("\n=== premise warnings (non-blocking) ===")
    for w, n in warn.most_common():
        print(f"  {w:40s}: {n}")

sample         used  verif supp  mode             opt_fol cause
---------------------------------------------------------------
T1_0000_00     False True  False fewest_premise   4       QUERY_MODE_DEFERRED:fewest_premise
T1_0000_01     True  True  True  entailment       0       — (solved)
T1_0001_00     False True  False strongest_conclusion 4       QUERY_MODE_DEFERRED:strongest_conclusion
T1_0001_01     True  True  True  entailment       0       Z3_TRUE_UNCERTAIN
T1_0002_00     True  True  True  entailment       4       Z3_TRUE_UNCERTAIN
T1_0002_01     True  True  True  entailment       0       Z3_TRUE_UNCERTAIN
T1_0003_00     True  True  True  entailment       4       Z3_TRUE_UNCERTAIN
T1_0003_01     True  True  True  entailment       0       Z3_TRUE_UNCERTAIN
T1_0004_00     True  True  True  entailment       4       Z3_TRUE_UNCERTAIN
T1_0004_01     True  True  True  entailment       0       Z3_TRUE_UNCERTAIN
T1_0005_00     True  True  True  entailment       4       Z3_TRUE_UNCERTAIN

## Notes
- Set `N_TYPE1 = None` to eval all 808 type-1 questions; `N_TYPE2 = None` for all 100 type-2 samples.
- Type 1 polar questions send `options=["Yes","No","Uncertain"]` so the server knows expected answer labels.
- Type 2 returns a mock response (`answer="Unknown"`) until the physics pipeline is merged.
- `solver_used=False` means premise verification failed or the question was unsupported — answer is the uncertain token.
- To inspect one result: `next(r for r in results if r['query_id'] == 'T1_0000_00')['_response']`.